In [21]:
import os
if os.path.basename(os.getcwd()) == 'Demonstrations':
    os.chdir('..')

<div style="background-color: #51daca; color: white; padding: 30px; border-radius: 0px;">
<h1 style="margin: 0;  color: #04335a">Convergence Test for the Stokes Equations </h3>
</div>

Imports:

In [22]:
from scipy.sparse.linalg import spsolve
import pandas as pd

from Utilities.Stokes_felib import *
from Utilities.Mesh_processing import *
from Solvers.Exchanger_Device import compute_U_P_solution

<div style="background-color: #7ac4ef; color: white; padding: 10px; border-radius: 0px;">
<h3 style="margin: 0; color: #11116e">Convergence Analysis via Reference Solutions on an Exchanger Device</h3>
</div>

Reference Solution computation:

In [24]:
data = np.load('Meshes/exchanger_device_altered_mesh_data.npz')
p_raw = data['p']
tri_idx = data['t_raw']
data.close()

p_base, e_base, t_base = build_stable_mesh(p_raw, tri_idx)
p_coarse_ref, e_coarse_ref, t_coarse_ref = refine_n_times(p_base, e_base, t_base, number_of_refinements=5)
p_fine_ref, e_fine_ref, t_fine_ref = refine(p_coarse_ref, e_coarse_ref, t_coarse_ref)

In [ ]:
ux_ref, uy_ref, p_ref = compute_U_P_solution(p_fine_ref, t_fine_ref, e_fine_ref, p_coarse_ref, t_coarse_ref)
reference_solution = [ux_ref, uy_ref, p_ref]

save_simulation_data(p_fine_ref, e_fine_ref, t_fine_ref,
                     p_coarse_ref, e_coarse_ref, t_coarse_ref,
                     ux_ref, uy_ref, p_ref,
                     name='Reference_Solution')

K is of the shape (3457507, 3457507)


Error table generation:


| $h$  | $(\|\|u_x - u_{x,ref}\|\|)$ | $(\|\|u_y - u_{y,ref}\|\|)$ | $(\|\|p - p_{ref}\|\|)$ |
| :--- | :-------------------------- | :-------------------------- | :---------------------- |
| 1/8  | $\epsilon_{u_x,1}$          | $\epsilon_{u_y,1}$          | $\epsilon_{p,1}$        |
| 1/16 | $\epsilon_{u_x,2}$          | $\epsilon_{u_y,2}$          | $\epsilon_{p,2}$        |
| 1/32 | $\epsilon_{u_x,3}$          | $\epsilon_{u_y,3}$          | $\epsilon_{p,3}$        |


In [ ]:
error_ux = []
error_uy = []
error_u = []
error_p = []
refinements = []

p_original, e_original, t_original = build_stable_mesh(p_raw, tri_idx)

for i in range(1,5):
    p_coarse, e_coarse, t_coarse = refine_n_times(p_original, e_original, t_original, 
                                                  number_of_refinements=i)
    p_fine, e_fine, t_fine = refine(p_coarse, e_coarse, t_coarse)
    ux, uy, p_sol = compute_U_P_solution(p_fine, t_fine, e_fine, p_coarse, t_coarse)

    save_simulation_data(p_fine, e_fine, t_fine,
                         p_coarse, e_coarse, t_coarse,
                         ux, uy, p_sol,
                         name=f'Convergence_Step_{i}')
    
    ux_ref_interp = griddata((p_fine_ref[:, 0], p_fine_ref[:, 1]), ux_ref, (p_fine[:, 0], p_fine[:, 1]), method='cubic')
    uy_ref_interp = griddata((p_fine_ref[:, 0], p_fine_ref[:, 1]), uy_ref, (p_fine[:, 0], p_fine[:, 1]), method='cubic')
    p_ref_interp  = griddata((p_coarse_ref[:, 0], p_coarse_ref[:, 1]), p_ref, (p_coarse[:, 0], p_coarse[:, 1]), method='cubic')

    ux_uxref = np.linalg.norm(ux - ux_ref_interp)
    uy_uyref = np.linalg.norm(uy - uy_ref_interp)
    u_uref   = np.sqrt(ux_uxref**2 + uy_uyref**2)
    p_pref   = np.linalg.norm(p_sol - p_ref_interp)
    
    error_ux.append(ux_uxref)
    error_uy.append(uy_uyref)
    error_u.append(u_uref)
    error_p.append(p_pref)
    refinements.append(i)


SyntaxError: incomplete input (3450133673.py, line 3)

In [ ]:
data_dict = {
    'Refinement Level': refinements,
    'Error ux': error_ux,
    'Error uy': error_uy,
    'Error u (Total)': error_u,
    'Error p': error_p
}

df_errors = pd.DataFrame(data_dict)

df_errors['Velocity Convergence Rate'] = np.log(df_errors['Error u (Total)'].shift(1) / df_errors['Error u (Total)']) / np.log(2)
df_errors['Pressure Convergence Rate'] = np.log(df_errors['Error p'].shift(1) / df_errors['Error p']) / np.log(2)

pd.set_option('display.float_format', lambda x: f'{x:.4e}' if x < 1 else f'{x:.2f}')
print(df_errors.to_string(index=False))